# ISIC 2019 — EfficientNet-B0 Architecture

[PyTorch Data Tutorial](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) 
https://isic-archive.s3.amazonaws.com/challenges/2019


ISIC 2019 data is downloaded and placed under `DATA_ROOT` below.

In [1]:
!pip install timm -q

In [2]:
#IGNORE---TESTING; pre-download weights and cache EfficientNet-B0 weights for later use in build_model()
#import timm
#timm.create_model("efficientnet_b0", pretrained=True)
#print("Weights cached!")

In [3]:
import os, json, time, warnings
import wandb
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn

import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR #baseline
from torchvision import transforms #for iamge preprocessing, e.g. resize, augmentation etc

from sklearn.model_selection import GroupShuffleSplit #for lesion_id split later

warnings.filterwarnings("ignore")
print("Imports OK")

Imports OK


In [4]:
import subprocess, zipfile

# PATHS 
DATA_ROOT = "/scratch/umw7eg/isic2019"
CKPT_DIR  = "/scratch/umw7eg/isic2019/checkpoints"
os.makedirs(DATA_ROOT, exist_ok=True)

BASE_URL = "https://isic-archive.s3.amazonaws.com/challenges/2019"

In [ ]:
# Download CSVs & zips 
DOWNLOADS = [
    (f"{BASE_URL}/ISIC_2019_Training_GroundTruth.csv", f"{DATA_ROOT}/train_gt.csv"),
    (f"{BASE_URL}/ISIC_2019_Training_Metadata.csv",    f"{DATA_ROOT}/train_meta.csv"),
    (f"{BASE_URL}/ISIC_2019_Test_GroundTruth.csv",     f"{DATA_ROOT}/test_gt.csv"),
    
    (f"{BASE_URL}/ISIC_2019_Training_Input.zip",       f"{DATA_ROOT}/train_imgs.zip"),
    (f"{BASE_URL}/ISIC_2019_Test_Input.zip",           f"{DATA_ROOT}/test_imgs.zip"),
]

for url, dest in DOWNLOADS:
    print(f"Downloading {os.path.basename(dest)} ...")
    subprocess.run(
        ["curl", "--location", "--progress-bar",
         "--retry", "3", "--retry-delay", "5",
         "--output", dest, url],
        check=True,
    )

In [ ]:
# Unzip
for zip_path, extract_dir in [
    (f"{DATA_ROOT}/train_imgs.zip", DATA_ROOT),
    (f"{DATA_ROOT}/test_imgs.zip",  DATA_ROOT),
]:
    print(f"Extracting {os.path.basename(zip_path)} ...")
    
    # Sanity check
    size_mb = os.path.getsize(zip_path) / (1024**2)
    print(f"  File size: {size_mb:.1f} MB")
    if size_mb < 1:
        raise RuntimeError(f"File too small — download likely failed: {zip_path}")
    
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    
    os.remove(zip_path)
    print(f"  Done & removed {os.path.basename(zip_path)}")

In [5]:
# Path 
TRAIN_DIR  = os.path.join(DATA_ROOT, "ISIC_2019_Training_Input")
TEST_DIR   = os.path.join(DATA_ROOT, "ISIC_2019_Test_Input")
TRAIN_CSV  = os.path.join(DATA_ROOT, "train_gt.csv")
TEST_CSV   = os.path.join(DATA_ROOT, "test_gt.csv")
TRAIN_META = os.path.join(DATA_ROOT, "train_meta.csv")

print("Done! Data ready.")

Done! Data ready.


## Config for ablation studies

In [6]:
#os.makedirs(CKPT_DIR, exist_ok=True)

Config = {
    # model
    "architecture": "efficientnet_b0",  # "resnet50" | "mobilenetv3_small"
    "pretrained":   True,              # False = train from scratch ablation
    "freeze_bb":    False,             # True = head-only training ablation
    "loss_fn":      "weighted_ce",      # "ce" | "focal"
    "augmentation": "standard",           # "none" | "geometric" | "color" | "standard"
    "img_size":     224,
    "batch_size":   32,
    "epochs":       30,
    "lr":           1e-4,
    "val_split":    0.20,
    "patience":     10,                #stop epoch if no improvement for 10
    "seed":         42,                #set_seed for reproducibility 
    "classes":      ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"],
}
Config["num_classes"] = len(Config["classes"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Config: {Config}")

Device: cuda
Config: {'architecture': 'efficientnet_b0', 'pretrained': True, 'freeze_bb': False, 'loss_fn': 'weighted_ce', 'augmentation': 'standard', 'img_size': 224, 'batch_size': 32, 'epochs': 30, 'lr': 0.0001, 'val_split': 0.2, 'patience': 10, 'seed': 42, 'classes': ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC'], 'num_classes': 8}


In [7]:
import random
def set_seed(seed=Config["seed"]):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Seed set to {Config['seed']}")

Seed set to 42


In [8]:
#Dataset 

class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(f"{self.img_dir}/{row['image']}.jpg").convert("RGB")
        
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label

def get_transforms(split):
    aug = Config["augmentation"]
    
    if split in ("val", "test"):
        return transforms.Compose([
            transforms.Resize((Config["img_size"], Config["img_size"])),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    
    # Train transforms vary by augmentation setting in the Config code cell
    if aug == "none":
        aug_tfms = []
    elif aug == "geometric":
        aug_tfms = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(degrees=15),
        ]
    elif aug == "color":
        aug_tfms = [
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        ]
    elif aug == "standard":
        aug_tfms = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomRotation(degrees=15),
        ]
    
    return transforms.Compose([
        transforms.Resize((Config["img_size"], Config["img_size"])),
        *aug_tfms,
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

In [9]:
import timm #using timm library as per Prof feedback
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

def build_model(num_classes=Config["num_classes"]):    
    model = timm.create_model("efficientnet_b0",
                              pretrained=Config["pretrained"],
                              num_classes=num_classes)
    return model

model = build_model().to(device)

print(type(model))
print("Model loaded OK")

<class 'timm.models.efficientnet.EfficientNet'>
Model loaded OK


## Training loop

In [10]:
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [22]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)                        
            probs   = F.softmax(outputs, dim=1)
            preds   = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())       

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    acc  = (all_preds == all_labels).mean()
    bacc = balanced_accuracy_score(all_labels, all_preds)

    # sensitivity & specificity per class
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(Config["num_classes"])))
    sensitivity, specificity, f1_per = [], [], []
    for i in range(Config["num_classes"]):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sensitivity.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_per.append(2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0)

    # AUC per class
    auc_per_class = {}
    for i, cls in enumerate(Config["classes"]):
        try:
            auc_per_class[cls] = roc_auc_score((all_labels == i).astype(int), all_probs[:, i])
        except ValueError:
            auc_per_class[cls] = float("nan")

    return acc, bacc, np.array(sensitivity), np.array(specificity), np.array(f1_per), auc_per_class

In [23]:
#training loop 

#def train_model(train_df, val_df, num_classes):
#    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#    train_ds = ISICDataset(train_df, CONFIG["data_root"], train_tfms)
#    val_ds = ISICDataset(val_df, CONFIG["data_root"], val_tfms)

#    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=4, pin_memory=True)
#    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=4, pin_memory=True)

#    model = build_model(num_classes).to(device)

#    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
#    criterion = nn.CrossEntropyLoss()

#    best_acc = 0
#    patience = 10
#    no_improve = 0

#    for epoch in range(CONFIG["epochs"]):
#        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
#        val_acc = evaluate(model, val_loader, device)

#        print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Val Acc={val_acc:.4f}")

#        if val_acc > best_acc:
#            best_acc = val_acc
#            no_improve = 0
#            torch.save(model.state_dict(), "best_model.pth")
#        else:
#            no_improve += 1

#        if no_improve >= patience:
#            print("Early stopping triggered")
#            break

#    return model

In [24]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Convert label column
train_df["label"] = train_df[Config["classes"]].values.argmax(axis=1)
test_df["label"]  = test_df[Config["classes"]].values.argmax(axis=1)

# Train/val split stratified by label...to be improved later by adding patient-level split with lesion_id(GroupShuffleSplit)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    train_df,
    test_size=Config["val_split"],
    stratify=train_df["label"],
    random_state=Config["seed"],
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 20264 | Val: 5067 | Test: 8238


In [25]:
#show class imabalnce in frequency table
counts        = train_df["label"].value_counts().sort_index()
CLASS_FREQ    = (counts / counts.sum()).to_dict()

print("Class frequencies:")

for name, df_ in [("TRAIN", train_df), ("VAL", val_df)]:
    labels = df_[Config["classes"]].values.argmax(axis=1)
    cnts   = np.bincount(labels, minlength=len(Config["classes"]))

    print(f"\n{name} ({len(df_)} images):")
    for i, c in enumerate(Config["classes"]):
        print(f"  {c:6s}: {cnts[i]:5d}  ({100*cnts[i]/len(df_):4.1f}%)")

Class frequencies:

TRAIN (20264 images):
  MEL   :  3618  (17.9%)
  NV    : 10300  (50.8%)
  BCC   :  2658  (13.1%)
  AK    :   694  ( 3.4%)
  BKL   :  2099  (10.4%)
  DF    :   191  ( 0.9%)
  VASC  :   202  ( 1.0%)
  SCC   :   502  ( 2.5%)

VAL (5067 images):
  MEL   :   904  (17.8%)
  NV    :  2575  (50.8%)
  BCC   :   665  (13.1%)
  AK    :   173  ( 3.4%)
  BKL   :   525  (10.4%)
  DF    :    48  ( 0.9%)
  VASC  :    51  ( 1.0%)
  SCC   :   126  ( 2.5%)


In [26]:
# 10% subset TO REMOVE LATER
train_df = train_df.sample(frac=0.01, random_state=Config["seed"]).reset_index(drop=True)
val_df   = val_df.sample(frac=0.01,   random_state=Config["seed"]).reset_index(drop=True)
test_df  = test_df.sample(frac=0.01,  random_state=Config["seed"]).reset_index(drop=True)
print(f"Subset — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

# Datasets
train_ds = ISICDataset(train_df, TRAIN_DIR, get_transforms("train"))
val_ds   = ISICDataset(val_df,   TRAIN_DIR, get_transforms("val"))
test_ds  = ISICDataset(test_df,  TEST_DIR,  get_transforms("test"))

# Dataloaders
train_loader = DataLoader(
    train_ds,
    batch_size=Config["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=Config["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=Config["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

Subset — train: 203, val: 51, test: 82


In [27]:
# compute class weights from standard ce so we can address class imbalance
counts = train_df["label"].value_counts().sort_index()
class_weights = (1.0 / counts).values
class_weights = class_weights / class_weights.sum() * len(class_weights)  # normalize
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

# build loss from Config
if Config["loss_fn"] == "weighted_ce":
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
elif Config["loss_fn"] == "ce":
    loss_fn = nn.CrossEntropyLoss()
else:
    raise ValueError(f"Unknown loss_fn: {Config['loss_fn']}")

In [28]:
#Q: DID U RUN WANDB LOGIN IN TERMINAL?

In [29]:
#wandb.login(key="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")
#run the below for W&B login
#!/home/umw7eg/.local/bin/wandb login
#wandb.login(key="API_key")

os.environ["WANDB_NOTEBOOK_NAME"] = "EfficientNet_B0.ipynb"

wandb.init(
    entity="umw7eg_uva",
    project="ISIC2019",
    name=f"{Config['architecture']}_{Config['loss_fn']}_{Config['augmentation']}_pretrained{Config['pretrained']}",
    config={
        "architecture":  Config["architecture"],
        "loss_fn":       Config["loss_fn"],
        "augmentation":  Config["augmentation"],
        "pretrained":    Config["pretrained"],
        "freeze_bb":     Config["freeze_bb"],
        "img_size":      Config["img_size"],
        "batch_size":    Config["batch_size"],
        "epochs":        Config["epochs"],
        "lr":            Config["lr"],
        "val_split":     Config["val_split"],
        "patience":      Config["patience"],
        "seed":          Config["seed"],
    }
)

In [30]:
model     = build_model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=Config["lr"])
scheduler = CosineAnnealingLR(optimizer, T_max=Config["epochs"], eta_min=1e-6)

best_bacc, best_epoch, no_improve = -1, 0, 0
history   = []
ckpt_path = os.path.join(CKPT_DIR, "efficientnet_b0_best.pt")

print(f"Training | pretrained={Config['pretrained']} | loss={Config['loss_fn']}\n")
t0 = time.time()

for epoch in range(1, Config["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    acc, bacc, sensitivity, specificity, f1_per, auc_per_class = evaluate(model, val_loader, device)

    auc_macro = np.nanmean(list(auc_per_class.values()))
    
    log_dict = {
        "epoch":                 epoch,
        "train/loss":            train_loss,
        "val/acc":               acc,
        "val/bacc":              bacc,
        "val/sensitivity_macro": sensitivity.mean(),
        "val/specificity_macro": specificity.mean(),
        "val/f1_macro":          f1_per.mean(),
        "val/auc_macro":         auc_macro,
    }
    for i, cls in enumerate(Config["classes"]):
        log_dict[f"val/sensitivity_{cls}"] = sensitivity[i]
        log_dict[f"val/specificity_{cls}"] = specificity[i]
        log_dict[f"val/f1_{cls}"]          = f1_per[i]
        log_dict[f"val/auc_{cls}"]         = auc_per_class[cls]   # per-class AUC

    wandb.log(log_dict)
    
    history.append({
        "epoch":      epoch,
        "train_loss": train_loss,
        
        "val_acc":    acc,
        "val_bacc":   bacc,
        
        "val_sens":   float(sensitivity.mean()),
        "val_spec":   float(specificity.mean()),
        
        "val_f1":     float(f1_per.mean()),
        "val_auc":    float(auc_macro),
    })

    print(f"Epoch {epoch:3d}/{Config['epochs']} | loss={train_loss:.4f} | "
          f"acc={acc:.4f} | bacc={bacc:.4f} | "
          f"sens={sensitivity.mean():.4f} | spec={specificity.mean():.4f} | "
          f"f1={f1_per.mean():.4f} | auc={auc_macro:.4f}", end="")
    
    scheduler.step()

    if bacc > best_bacc:
        best_bacc, best_epoch = bacc, epoch
        no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
        print("saved")
    else:
        no_improve += 1
        print(f"  (no improve {no_improve}/{Config['patience']})")
        if no_improve >= Config["patience"]:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nDone in {(time.time()-t0)/60:.1f} min")

print(f"Best   | epoch={best_epoch} | bacc={best_bacc:.4f}")
print(f"Final  | acc={acc:.4f} | bacc={bacc:.4f} | "
      f"sens={sensitivity.mean():.4f} | spec={specificity.mean():.4f} | "
      f"f1={f1_per.mean():.4f} | auc={auc_macro:.4f}")

history_path = os.path.join(CKPT_DIR, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved to {history_path}")

Training | pretrained=True | loss=weighted_ce

Epoch   1/30 | loss=4.1620 | acc=0.0588 | bacc=0.0434 | sens=0.0271 | spec=0.8701 | f1=0.0349 | auc=0.5261saved
Epoch   2/30 | loss=2.8351 | acc=0.0784 | bacc=0.0508 | sens=0.0317 | spec=0.8745 | f1=0.0413 | auc=0.5018saved
Epoch   3/30 | loss=2.6839 | acc=0.0980 | bacc=0.0794 | sens=0.0496 | spec=0.8759 | f1=0.0568 | auc=0.5014saved
Epoch   4/30 | loss=1.7574 | acc=0.1569 | bacc=0.1862 | sens=0.1164 | spec=0.8800 | f1=0.0990 | auc=0.5112saved
Epoch   5/30 | loss=1.2738 | acc=0.1765 | bacc=0.2044 | sens=0.1278 | spec=0.8841 | f1=0.1142 | auc=0.5306saved
Epoch   6/30 | loss=1.2556 | acc=0.2157 | bacc=0.2408 | sens=0.1505 | spec=0.8876 | f1=0.1352 | auc=0.5496saved
Epoch   7/30 | loss=1.0261 | acc=0.2941 | bacc=0.3238 | sens=0.2024 | spec=0.8975 | f1=0.1722 | auc=0.5643saved
Epoch   8/30 | loss=0.9634 | acc=0.3137 | bacc=0.3100 | sens=0.1938 | spec=0.9001 | f1=0.1778 | auc=0.5726  (no improve 1/10)
Epoch   9/30 | loss=0.9088 | acc=0.3137 | b

***!!IGNORE THE LINES BELOW FOR NOW***

In [32]:
def print_benchmark(model, label="Model"):
    params_total     = sum(p.numel() for p in model.parameters())
    params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb          = params_total * 4 / 1024 ** 2  # float32

    # Inference speed
    model.eval()
    dummy = torch.randn(1, 3, 224, 224).to(device)
    with torch.no_grad():
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(50):
            model(dummy)
        end.record()
        torch.cuda.synchronize()
    inference_ms = start.elapsed_time(end) / 50

    # GFLOPs (requires thop)
    try:
        from thop import profile
        flops, _ = profile(model, inputs=(dummy,), verbose=False)
        gflops = flops / 1e9
    except:
        gflops = None

    bench = {
        "params_total":     params_total,
        "params_trainable": params_trainable,
        "size_mb":          round(size_mb, 2),
        "inference_ms":     round(inference_ms, 2),
        "gflops":           round(gflops, 4) if gflops else None
    }

    print(f"\n=== Benchmark: {label} ===")
    for k, v in bench.items():
        print(f"  {k}: {v}")

    return bench
        
model.load_state_dict(torch.load(ckpt_path, map_location=device))
bench = print_benchmark(model, label=f"EfficientNet-B0 ({Config['loss_fn']}, aug={Config['augmentation']})")
wandb.summary["benchmark/params_total"]     = bench["params_total"]
wandb.summary["benchmark/params_trainable"] = bench["params_trainable"]
wandb.summary["benchmark/size_mb"]          = bench["size_mb"]
wandb.summary["benchmark/inference_ms"]     = bench["inference_ms"]
wandb.summary["benchmark/gflops"]           = bench["gflops"]
wandb.summary["best_val_bacc"]              = best_bacc
wandb.summary["best_epoch"]                 = best_epoch
artifact = wandb.Artifact(
    name=f"efficientnet_b0_{Config['loss_fn']}_{Config['augmentation']}",
    type="model",
    description=f"Best checkpoint — val BACC {best_bacc:.4f} @ epoch {best_epoch}"
)
artifact.add_file(ckpt_path)
wandb.log_artifact(artifact)
print("Checkpoint artifact logged to W&B")
wandb.finish()
print("W&B run finished.")

# History saving
history_path = os.path.join(CKPT_DIR, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved to {history_path}")


=== Benchmark: EfficientNet-B0 (weighted_ce, aug=standard) ===
  params_total: 4017796
  params_trainable: 4017796
  size_mb: 15.33
  inference_ms: 8.23
  gflops: None
Checkpoint artifact logged to W&B


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▆▅▄▃▃▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▂▃▃▄▅▆▆▆▆▇▇▆▇▇▇█▇██▇▇█▇█▇▇██
val/auc_BCC,▁▄▇▇██████▇▇█▇███▇████████▇█▇▇
val/auc_BKL,█▄▂▁▁▁▂▃▃▃▃▃▄▄▅▅▄▃▄▄▄▄▄▅▄▅▅▄▄▄
val/auc_MEL,▁▅▅▆▇▇█▇█▇█▇▇▇▇▇▇██▇▇███▇▆▇▇▇█
val/auc_NV,▄▂▁▂▃▄▅▆▆▆▆▇▇▇▇▇██████▇███████
val/auc_SCC,█▄▂▂▁▂▁▁▁▂▁▂▂▂▂▂▂▂▂▃▃▃▃▂▃▃▂▃▃▂
val/auc_macro,▂▁▁▂▃▄▄▅▅▆▅▆▇▆▇▇▇▇▇▇█████▇▇██▇
val/bacc,▁▁▂▄▄▅▇▆▆▇▇▇▇▇██▇▇▇██▇▇███▇███
+30,...


W&B run finished.
History saved to /scratch/umw7eg/isic2019/checkpoints/history.json
